# TRACK-FA Entropy and Mutual-Information Feature Selection

This notebook uses mutual information only for visit/progression separation. Clinical scores are kept out of model training and feature selection targets.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project imports and data paths from either the repo root or notebooks folder.
def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


_REPO_ROOT = find_project_root(Path.cwd())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from src.config import set_global_seeds  # noqa: E402
from src.data.qc import standardize_train_test  # noqa: E402
from src.eval.cv import lda_loocv  # noqa: E402
from src.eval.metrics import (  # noqa: E402
    bootstrap_ci_d,
    compute_cohens_d,
    compute_srm,
    paired_deltas_from_long,
)
from src.features.entropy import (  # noqa: E402
    mi_feature_vs_binary_label,
    rank_features_by_mi,
)
from src.features.selection import (  # noqa: E402
    _global_rank,
    make_selection_fn,
    select_topk_global,
    select_topk_by_group,
)


## Load dependencies


In [ ]:
set_global_seeds(42)

from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long

RANDOM_SEED = 42
N_BOOT = 2000  # Number of bootstrap resamples for confidence intervals.
K_VALUES = [1, 2, 3, 5]

DATA_PATH = _REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
# Load TRACK-FA paired rows, infer feature groups, and convert to visit-level rows.
pairs = pd.read_csv(DATA_PATH)
groups = infer_trackfa_feature_groups(pairs)
df_long = trackfa_pairs_to_long(pairs)
subject_col = "pair_id"

background = list(groups.background)
structural = list(groups.poms + groups.brainspinemorph)
structural_ext = list(structural)
diffusion = list(groups.braindti)

FEATURE_SETS = {
    "background": background,
    "structural": structural,
    "structural_ext": structural_ext,
    "diffusion": diffusion,
    "background_structural": background + structural,
    "background_structural_ext": background + structural_ext,
    "background_diffusion": background + diffusion,
    "structural_diffusion": structural + diffusion,
    "structural_ext_diffusion": structural_ext + diffusion,
    "background_structural_diffusion": background + structural + diffusion,
    "background_structural_ext_diffusion": background + structural_ext + diffusion,
}

ALL_FEATURES = sorted(set(background + structural_ext + diffusion))

# Reporting groups for mutual-information tables.
FEATURE_GROUPS = {
    "background": background,
    "structural": structural,
    "structural_ext": structural_ext,
    "diffusion": diffusion,
}

paired_subjects = df_long.groupby(subject_col)["visit"].nunique()
paired_subjects = paired_subjects[paired_subjects == 2].index

print("Pairs shape:", pairs.shape)
print("Long shape:", df_long.shape)
print("Pair column:", subject_col, "| n_pairs:", df_long[subject_col].nunique())
print("Feature counts:", {k: len(v) for k, v in FEATURE_GROUPS.items()})
print("Clinical scores retained only for benchmarks:", [c for c in ["FARS", "SARA", "ADL"] if c in df_long.columns])
print("Paired rows for progression:", len(paired_subjects))


## Score single-feature progression


In [ ]:
feature_memberships = {}
for g, feats in FEATURE_GROUPS.items():
    for f in feats:
        feature_memberships.setdefault(f, set()).add(g)

features_for_mi = ALL_FEATURES

mi_visit_rows = []
visit_y = (df_long["visit"].values == 2).astype(int)
for f in features_for_mi:
    if f not in df_long.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_binary_label(df_long[f], visit_y, is_discrete=(f == "sex"))
    mi_visit_rows.append({"feature": f, "mi_visit": mi, "n_visit": n})
mi_df = pd.DataFrame(mi_visit_rows)

rows_e = []
for _, r in mi_df.iterrows():
    f = r["feature"]
    groups = sorted(feature_memberships.get(f, []))
    if not groups:
        continue
    for g in groups:
        rows_e.append({
            "feature": f, "group": g,
            "mi_visit": r.get("mi_visit"), "n_visit": r.get("n_visit"),
        })
single_feature_entropy_df = pd.DataFrame(rows_e)
display(single_feature_entropy_df)


## Estimate mutual information


In [4]:
single_rows = []
paired = df_long[df_long[subject_col].isin(paired_subjects)].copy()
for f in ALL_FEATURES:
    if f not in paired.columns:
        continue
    tmp = paired[[subject_col, "visit", f]].dropna().copy()
    if tmp.empty:
        continue
    deltas = paired_deltas_from_long(tmp.rename(columns={f: "value"}), subject_col, "visit", "value")
    d_out = compute_cohens_d(deltas)
    srm_out = compute_srm(deltas)
    tmp_oof = tmp.rename(columns={f: "value"})
    _, d_lo, d_hi = bootstrap_ci_d(tmp_oof, subject_col, "visit", "value", n_boot=N_BOOT, seed=RANDOM_SEED)
    if f in background:
        g = "background"
    elif f in structural:
        g = "structural"
    elif f in structural_ext:
        g = "structural_ext"
    elif f in diffusion:
        g = "diffusion"
    else:
        g = "unknown"
    single_rows.append({
        "feature": f, "group": g,
        "d_feature": d_out["d"], "srm_feature": srm_out["srm"],
        "d_ci_low": d_lo, "d_ci_high": d_hi,
        "n_subjects": d_out["n"],
        "mean_diff": d_out["mean"], "sd_diff": d_out["sd"],
    })
single_feature_d_df = pd.DataFrame(single_rows).sort_values("d_feature", ascending=False)
display(single_feature_d_df)


,feature,group,d_feature,srm_feature,d_ci_low,d_ci_high,n_subjects,mean_diff,sd_diff
150,x3rd_Ventricle,structural,0.389207,0.389207,0.278132,0.519447,207,21.910314,56.294723
151,x4th_Ventricle,structural,0.385422,0.385422,0.253905,0.525049,207,35.902556,93.151180
66,Lateral_Ventricle,structural,0.385271,0.385271,0.289335,0.528855,207,382.529111,992.882623
116,RD_SCP,structural,0.260127,0.260127,0.122014,0.393669,207,0.000011,0.000041
84,MD_SCP,structural,0.233416,0.233416,0.094469,0.368001,207,0.000008,0.000036
...,...,...,...,...,...,...,...,...,...
139,disease_duration,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000
141,gaa_1,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000
142,gaa_2,background,NaN,NaN,NaN,NaN,202,0.000000,0.000000
143,onset_age,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000


## Register entropy-selected feature sets


In [5]:
registry_rows = []
for task, source in [("lda_visit", "mi_visit"), ("reg_fars1", "mi_fars1"), ("reg_dfars", "mi_dfars")]:
    global_rank = _global_rank(single_feature_entropy_df, source)
    for k in K_VALUES:
        k2 = min(int(k), len(global_rank))
        registry_rows.append({
            "task": task,
            "selection_mode": "entropy_topk_global",
            "entropy_source": source,
            "k_selected": k2,
            "selected_features": str(global_rank[:k2]),
        })
    group_to_rank = {g: _global_rank(single_feature_entropy_df, source, group=g) for g in FEATURE_GROUPS.keys()}
    for k in K_VALUES:
        selected = []
        for g, rank in group_to_rank.items():
            selected.extend(rank[: min(int(k), len(rank))])
        seen = set()
        sel = []
        for f in selected:
            if f not in seen:
                seen.add(f)
                sel.append(f)
        registry_rows.append({
            "task": task,
            "selection_mode": "entropy_topk_group",
            "entropy_source": source,
            "k_selected": int(k),
            "selected_features": str(sel),
        })
entropy_selected_feature_sets_df = pd.DataFrame(registry_rows)
display(entropy_selected_feature_sets_df)


,task,selection_mode,entropy_source,k_selected,selected_features
0,lda_visit,entropy_topk_global,mi_visit,1,['sMD_c3c5']
1,lda_visit,entropy_topk_global,mi_visit,2,"['sMD_c3c5', 'AD_ALIC']"
2,lda_visit,entropy_topk_global,mi_visit,3,"['sMD_c3c5', 'AD_ALIC', 'MD_SLF']"
3,lda_visit,entropy_topk_global,mi_visit,5,"['sMD_c3c5', 'AD_ALIC', 'MD_SLF', 'MD_mLEM', '..."
4,lda_visit,entropy_topk_group,mi_visit,1,"['age', 'sMD_c3c5', 'AD_ALIC']"
5,lda_visit,entropy_topk_group,mi_visit,2,"['age', 'disease_duration', 'sMD_c3c5', 'sAD_c..."
6,lda_visit,entropy_topk_group,mi_visit,3,"['age', 'disease_duration', 'gaa_1', 'sMD_c3c5..."
7,lda_visit,entropy_topk_group,mi_visit,5,"['age', 'disease_duration', 'gaa_1', 'gaa_2', ..."
8,reg_fars1,entropy_topk_global,mi_fars1,1,['FA_SCP']
9,reg_fars1,entropy_topk_global,mi_fars1,2,"['FA_SCP', 'disease_duration']"


## Train entropy-selected models


In [ ]:
results = []


def _make_sel(mode, k):
    return make_selection_fn(
        task="lda_visit", selection_mode=mode, k_selected=k,
        entropy_source="mi_visit",
        all_features=ALL_FEATURES,
        feature_groups=FEATURE_GROUPS,
    )


# LDA visit-separation models using complete feature sets.
for fs_name, feats in FEATURE_SETS.items():
    res = lda_loocv(df_long, feats, subject_col=subject_col, visit_col="visit", selection_fn=None)
    best_single_d = float(single_feature_d_df["d_feature"].max()) if len(single_feature_d_df) else np.nan
    results.append({
        "method": "LDA", "task": "separation", "target": "visit_axis",
        "feature_set": fs_name, "n_features": len(feats),
        "selection_mode": "full", "k_selected": np.nan, "entropy_source": np.nan,
        "d_score": res["d_score"], "srm": res["srm"],
        "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
        "rmse": np.nan, "r2": np.nan,
        "n_subjects": res["n_subjects"],
        "beats_best_single": (
            bool(res["d_score"] > best_single_d)
            if np.isfinite(res["d_score"]) and np.isfinite(best_single_d) else np.nan
        ),
        "notes": "",
    })

# LDA visit-separation models using entropy-selected features.
for selection_mode in ["entropy_topk_group", "entropy_topk_global"]:
    for k in K_VALUES:
        sel_fn = _make_sel(selection_mode, k)
        res = lda_loocv(df_long, ALL_FEATURES, subject_col=subject_col, visit_col="visit", selection_fn=sel_fn)
        best_single_d = float(single_feature_d_df["d_feature"].max()) if len(single_feature_d_df) else np.nan
        results.append({
            "method": "LDA", "task": "separation", "target": "visit_axis",
            "feature_set": "global_pool", "n_features": np.nan,
            "selection_mode": selection_mode, "k_selected": int(k), "entropy_source": "mi_visit",
            "d_score": res["d_score"], "srm": res["srm"],
            "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
            "rmse": np.nan, "r2": np.nan,
            "n_subjects": res["n_subjects"],
            "beats_best_single": (
                bool(res["d_score"] > best_single_d)
                if np.isfinite(res["d_score"]) and np.isfinite(best_single_d) else np.nan
            ),
            "notes": "entropy-selected (fold-specific)",
        })

# Single-feature reference rows for progression sensitivity.
if len(single_feature_d_df):
    best_by_d = single_feature_d_df.sort_values("d_feature", ascending=False).iloc[0]["feature"]
    best_by_mi_visit = single_feature_entropy_df.sort_values("mi_visit", ascending=False).iloc[0]["feature"]
    for feat, note in [(best_by_d, "single_feature_best_d"), (best_by_mi_visit, "single_feature_best_mi_visit")]:
        res = lda_loocv(df_long, [feat], subject_col=subject_col, visit_col="visit", selection_fn=None)
        best_single_d = float(single_feature_d_df["d_feature"].max())
        results.append({
            "method": "LDA", "task": "separation", "target": "visit_axis",
            "feature_set": feat, "n_features": 1,
            "selection_mode": "single_feature", "k_selected": 1, "entropy_source": np.nan,
            "d_score": res["d_score"], "srm": res["srm"],
            "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
            "rmse": np.nan, "r2": np.nan,
            "n_subjects": res["n_subjects"],
            "beats_best_single": (
                bool(res["d_score"] >= best_single_d)
                if np.isfinite(res["d_score"]) else np.nan
            ),
            "notes": note,
        })

results_df = pd.DataFrame(results)
print("Total result rows:", len(results_df))
display(results_df.sort_values("d_score", ascending=False).head(20))


## Final output summary


In [ ]:
print("Results rows:", len(results_df))
sep = results_df[results_df["task"] == "separation"].sort_values("d_score", ascending=False)
print("Top separation rows:", len(sep))
display(sep.head(10))
